# Entrenamiento inmunogenicidad — nested CV + LOPO + modelo final

Pipeline completo, **sin test fijo** (se usan todos los datos en el nested).

**Hiperparámetros — quién busca qué (importante):**
1. **Nested CV homology-aware** → en CADA fold externo se hace una búsqueda interna (Optuna) usando solo los otros folds. Los hiperparámetros pueden variar por fold. **De aquí sale el rendimiento honesto del método.**
2. **LOPO (leave-one-pathogen-out, por género)** → se corre con los hiperparámetros **del modelo final** (config fija). Es un estrés-test de generalización a un patógeno nuevo, no la estimación principal. (Para hacerlo totalmente anidado, ver nota al final.)
3. **Modelo final** → búsqueda de hiperparámetros sobre **todo** el dataset → reentrenar → guardar. Es el que se despliega; **no se evalúa sobre sí mismo** (su rendimiento esperado lo dan el nested CV y el LOPO).

**Corrección respecto a la versión anterior:** el early stopping usa una validación **interna del train**, nunca el fold/patógeno que se está evaluando (antes se usaba el grupo de test como `validation_data` → fuga suave).

**Salidas:** figuras (ROC, PR, AUC por patógeno, diagnóstico homología-vs-LOPO, barras de métricas) en `resultados/figuras/` (PNG 300 dpi + PDF), y CSVs en `resultados/` para rehacer figuras conjuntas a tu gusto.


In [ ]:
# 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Install Optuna + imports
!pip install optuna -q
import os, json, gc
import numpy as np, pandas as pd
import tensorflow.keras as ks, tensorflow as tf
import matplotlib.pyplot as plt
import optuna
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve, auc,
                             precision_recall_curve, precision_score, recall_score,
                             f1_score, matthews_corrcoef, balanced_accuracy_score,
                             confusion_matrix, classification_report)
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)
print('TF', tf.__version__)

In [ ]:
# 3. Configuration
NPZ_FILE        = '/content/drive/MyDrive/IMMUNO_MODEL/Bos_taurus/embeddings.npz'
MODEL_SAVE_PATH = '/content/drive/MyDrive/IMMUNO_MODEL/Bos_taurus/model_final.keras'

OUTDIR = os.path.join(os.path.dirname(NPZ_FILE), 'resultados')
FIGDIR = os.path.join(OUTDIR, 'figuras')
os.makedirs(FIGDIR, exist_ok=True)

N_TRIALS_NESTED = 15    # inner search per outer fold
N_TRIALS_FINAL  = 25    # search for the final model, over the whole dataset
print('Resultados ->', OUTDIR)

In [ ]:
# 4. Load the embeddings
data = np.load(NPZ_FILE, allow_pickle=True)
X_all   = data['X'].astype('float32')            # (N, 1280)
ids_all = data['ids'].astype(str)
y_all   = data['labels'].astype(int)             # 1 = immunogenic, 0 = not
cv_homology = data['cv_homology'].astype(str)    # fold_1..fold_5
cv_lopo     = data['cv_lopo'].astype(str)        # LOPO groups (genus/family)

print(f'Total: {X_all.shape}  pos={y_all.sum()}  neg={(y_all==0).sum()}')
print('\nHomology folds:')
for u in np.unique(cv_homology):
    m = cv_homology == u
    print(f'  {u}: n={m.sum()}  pos={y_all[m].sum()}')
print(f'\nLOPO: {len(np.unique(cv_lopo))} grupos')

In [ ]:
# 5. Model, training, and inner hyperparameter search
def build_model(cfg, norm):
    m = ks.Sequential([
        ks.layers.Input(shape=(X_all.shape[1],)),
        norm,
        ks.layers.Dense(1024, activation='relu'),
        ks.layers.Dropout(cfg['drop1']),
        ks.layers.Dense(256, activation='relu'),
        ks.layers.Dropout(cfg['drop2']),
        ks.layers.Dense(64, activation='relu'),
        ks.layers.Dense(1, activation='sigmoid'),
    ])
    m.compile(optimizer=ks.optimizers.Adam(cfg['lr']),
              loss=ks.losses.BinaryCrossentropy(), metrics=['AUC'])
    return m

def fit_model(Xtr, ytr, Xva, yva, cfg, epochs=500, patience=30):
    norm = ks.layers.Normalization(); norm.adapt(Xtr)
    classes = np.unique(ytr)
    cw = dict(zip(classes, compute_class_weight('balanced', classes=classes, y=ytr)))
    m = build_model(cfg, norm)
    m.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=epochs, batch_size=30,
          class_weight=cw, verbose=0,
          callbacks=[ks.callbacks.EarlyStopping(patience=patience, restore_best_weights=True)])
    return m

def inner_search(X, y, groups, n_trials, seed=SEED):
    # inner validation = a whole group of the train split (homology intact), NEVER the outer test
    val_g = np.unique(groups)[0]
    vi = groups == val_g; ti = ~vi
    def objective(trial):
        cfg = {'drop1': trial.suggest_float('drop1', 0.2, 0.8),
               'drop2': trial.suggest_float('drop2', 0.3, 0.9),
               'lr':    trial.suggest_float('lr', 1e-6, 1e-3, log=True)}
        m = fit_model(X[ti], y[ti], X[vi], y[vi], cfg)
        try:    a = roc_auc_score(y[vi], m.predict(X[vi], verbose=0).ravel())
        except ValueError: a = 0.0
        ks.backend.clear_session(); gc.collect()
        return a
    study = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=seed))
    study.optimize(objective, n_trials=n_trials)
    return study.best_params, study.best_value

In [ ]:
# 6. Homology-aware nested CV (inner search inside each outer fold)
def nested_cv(X, y, folds, pathogens, ids, n_trials=N_TRIALS_NESTED):
    oof_t, oof_s, oof_f, oof_p, oof_id = [], [], [], [], []
    per_fold, chosen = [], []
    for te_fold in np.unique(folds):
        te = folds == te_fold; tr = ~te
        cfg, iv = inner_search(X[tr], y[tr], folds[tr], n_trials)   # the outer fold is never touched
        tr_folds = folds[tr]; val_g = np.unique(tr_folds)[0]        # inner val, for early stopping
        vi = tr_folds == val_g
        Xtr, ytr = X[tr], y[tr]
        m = fit_model(Xtr[~vi], ytr[~vi], Xtr[vi], ytr[vi], cfg)
        s = m.predict(X[te], verbose=0).ravel()
        oof_t.append(y[te]); oof_s.append(s)
        oof_f.append(np.array([te_fold]*int(te.sum())))
        oof_p.append(pathogens[te]); oof_id.append(ids[te])
        both = len(np.unique(y[te])) > 1
        per_fold.append({'fold': te_fold, 'n': int(te.sum()), 'n_pos': int(y[te].sum()),
                         'auc_roc': roc_auc_score(y[te], s) if both else np.nan,
                         'auc_pr':  average_precision_score(y[te], s) if both else np.nan,
                         'cfg': json.dumps(cfg)})
        chosen.append(cfg)
        print(f'outer={te_fold}: inner-AUC={iv:.3f}  cfg={cfg}')
        ks.backend.clear_session(); gc.collect()
    return (np.concatenate(oof_t), np.concatenate(oof_s), np.concatenate(oof_f),
            np.concatenate(oof_p), np.concatenate(oof_id), pd.DataFrame(per_fold), chosen)

yt_hom, ys_hom, fold_hom, pat_hom, id_hom, perfold_hom, chosen_hom = \
    nested_cv(X_all, y_all, cv_homology, cv_lopo, ids_all)

print(f'\nNESTED CV homology-aware (OOF)  '
      f'AUC-ROC={roc_auc_score(yt_hom, ys_hom):.3f}  AUC-PR={average_precision_score(yt_hom, ys_hom):.3f}')
display(perfold_hom)

In [ ]:
# 7. FINAL MODEL - hyperparameters searched over the WHOLE dataset
final_cfg, final_iv = inner_search(X_all, y_all, cv_homology, N_TRIALS_FINAL)
print(f'Hiperparámetros del modelo final: {final_cfg}  (inner-AUC={final_iv:.3f})')

val_g = np.unique(cv_homology)[0]; vi = cv_homology == val_g   # inner val, early stopping only
final_model = fit_model(X_all[~vi], y_all[~vi], X_all[vi], y_all[vi], final_cfg)
final_model.save(MODEL_SAVE_PATH)
with open(os.path.join(OUTDIR, 'final_model_config.json'), 'w') as f:
    json.dump({**final_cfg, 'inner_auc': final_iv}, f, indent=2)
print(f'Modelo final guardado en {MODEL_SAVE_PATH}')
print('NOTA: el modelo final NO se evalúa sobre sí mismo; su rendimiento esperado')
print('      lo dan el nested CV (arriba) y el LOPO (abajo).')

In [ ]:
# 8. LOPO (leave-one-pathogen-out) with the final model's config
def lopo_fixed(X, y, groups, folds, pathogens, ids, cfg):
    oof_t, oof_s, oof_g, oof_id = [], [], [], []
    per_grp = []
    for g in np.unique(groups):
        te = groups == g; tr = ~te
        if te.sum() == 0 or tr.sum() == 0:
            continue
        tr_folds = folds[tr]; val_g = np.unique(tr_folds)[0]   # inner val, taken from the train split, not the held-out pathogen
        vi = tr_folds == val_g
        Xtr, ytr = X[tr], y[tr]
        m = fit_model(Xtr[~vi], ytr[~vi], Xtr[vi], ytr[vi], cfg)
        s = m.predict(X[te], verbose=0).ravel()
        oof_t.append(y[te]); oof_s.append(s)
        oof_g.append(np.array([g]*int(te.sum()))); oof_id.append(ids[te])
        both = len(np.unique(y[te])) > 1
        per_grp.append({'group': g, 'n': int(te.sum()), 'n_pos': int(y[te].sum()),
                        'auc_roc': roc_auc_score(y[te], s) if both else np.nan,
                        'auc_pr':  average_precision_score(y[te], s) if both else np.nan})
        ks.backend.clear_session(); gc.collect()
    return (np.concatenate(oof_t), np.concatenate(oof_s),
            np.concatenate(oof_g), np.concatenate(oof_id), pd.DataFrame(per_grp))

yt_lopo, ys_lopo, grp_lopo, id_lopo, perpath_lopo = \
    lopo_fixed(X_all, y_all, cv_lopo, cv_homology, cv_lopo, ids_all, final_cfg)

print(f'LOPO (OOF)  AUC-ROC={roc_auc_score(yt_lopo, ys_lopo):.3f}  '
      f'AUC-PR={average_precision_score(yt_lopo, ys_lopo):.3f}')
display(perpath_lopo.sort_values('n_pos', ascending=False))

In [ ]:
# 9. Per-pathogen diagnostic: homology-aware vs LOPO
#    Regroups the nested OOF predictions by the same pathogen group used in LOPO.
rows = []
for g in np.unique(cv_lopo):
    m = pat_hom == g
    if m.sum() == 0: continue
    both = len(np.unique(yt_hom[m])) > 1
    rows.append({'group': g, 'n': int(m.sum()), 'n_pos': int(yt_hom[m].sum()),
                 'auc_hom': roc_auc_score(yt_hom[m], ys_hom[m]) if both else np.nan,
                 'ap_hom':  average_precision_score(yt_hom[m], ys_hom[m]) if both else np.nan})
perpat_hom = pd.DataFrame(rows)

cmp = (perpath_lopo[['group','n','n_pos','auc_roc','auc_pr']]
       .rename(columns={'auc_roc':'auc_lopo','auc_pr':'ap_lopo'})
       .merge(perpat_hom[['group','auc_hom','ap_hom']], on='group', how='outer'))
cmp['gap_auc'] = cmp['auc_hom'] - cmp['auc_lopo']   # >0: learnt through homology, lost under LOPO
cmp = cmp.sort_values('gap_auc', ascending=False)
display(cmp)

In [ ]:
# 10. Save the CSVs, so the figures can be rebuilt later
def oof_metrics(yt, ys):
    yp = (ys > 0.5).astype(int)
    return {'auc_roc': roc_auc_score(yt, ys), 'auc_pr': average_precision_score(yt, ys),
            'precision': precision_score(yt, yp, zero_division=0),
            'recall': recall_score(yt, yp), 'f1': f1_score(yt, yp, zero_division=0),
            'mcc': matthews_corrcoef(yt, yp), 'bacc': balanced_accuracy_score(yt, yp)}

# 10a. per-sample OOF predictions (the one that matters for custom figures)
df_hom = pd.DataFrame({'scheme':'homology', 'id':id_hom, 'y_true':yt_hom, 'y_score':ys_hom,
                       'pathogen':pat_hom, 'homology_fold':fold_hom})
df_lop = pd.DataFrame({'scheme':'lopo', 'id':id_lopo, 'y_true':yt_lopo, 'y_score':ys_lopo,
                       'pathogen':grp_lopo, 'homology_fold':np.nan})
pd.concat([df_hom, df_lop], ignore_index=True).to_csv(os.path.join(OUTDIR,'oof_predictions.csv'), index=False)

# 10b. global metrics per scheme
summary = pd.DataFrame({'homology': oof_metrics(yt_hom, ys_hom),
                        'lopo':     oof_metrics(yt_lopo, ys_lopo)}).T
summary.to_csv(os.path.join(OUTDIR,'metrics_summary.csv'))

# 10c. metrics per fold / per pathogen / diagnostic / hyperparameters
perfold_hom.to_csv(os.path.join(OUTDIR,'perfold_homology.csv'), index=False)
perpath_lopo.to_csv(os.path.join(OUTDIR,'perpathogen_lopo.csv'), index=False)
cmp.to_csv(os.path.join(OUTDIR,'diagnostic_hom_vs_lopo.csv'), index=False)
pd.DataFrame([{'fold': r['fold'], **json.loads(r['cfg'])} for _, r in perfold_hom.iterrows()]) \
  .to_csv(os.path.join(OUTDIR,'chosen_hyperparams_homology.csv'), index=False)

print('CSVs guardados en', OUTDIR)
display(summary.round(3))

In [ ]:
# 11. FIGURES (saved as 300 dpi PNG + PDF)
def savefig(name):
    plt.savefig(os.path.join(FIGDIR, name+'.png'), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(FIGDIR, name+'.pdf'), bbox_inches='tight')

schemes = {'Nested CV homology-aware': (yt_hom, ys_hom),
           'LOPO (patógeno fuera)':    (yt_lopo, ys_lopo)}

# Fig 1 — ROC
plt.figure(figsize=(5,5))
for name,(yy,ss) in schemes.items():
    fpr,tpr,_ = roc_curve(yy,ss); plt.plot(fpr,tpr,lw=2,label=f'{name} (AUC={auc(fpr,tpr):.3f})')
plt.plot([0,1],[0,1],'k--',lw=1,alpha=.6)
plt.xlabel('1 - Especificidad (FPR)'); plt.ylabel('Sensibilidad (TPR)')
plt.title('ROC'); plt.legend(loc='lower right'); plt.tight_layout(); savefig('fig1_roc'); plt.show()

# Fig 2 — Precision-Recall
plt.figure(figsize=(5,5))
for name,(yy,ss) in schemes.items():
    pr,rc,_ = precision_recall_curve(yy,ss)
    plt.plot(rc,pr,lw=2,label=f'{name} (AP={average_precision_score(yy,ss):.3f})')
base = float(np.mean(y_all))
plt.axhline(base,ls='--',color='k',lw=1,alpha=.6,label=f'Azar (prevalencia={base:.2f})')
plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title('Precision-Recall')
plt.legend(loc='lower left'); plt.tight_layout(); savefig('fig2_pr'); plt.show()

# Fig 3 - AUC-ROC per pathogen (LOPO)
d = perpath_lopo.dropna(subset=['auc_roc']).sort_values('auc_roc')
plt.figure(figsize=(7, max(3, 0.45*len(d))))
plt.barh(d['group'], d['auc_roc'], color='#1f77b4')
plt.axvline(0.5, color='gray', ls='--')
plt.axvline(roc_auc_score(yt_lopo, ys_lopo), color='red', ls=':', label='OOF global')
plt.xlabel('AUC-ROC'); plt.title('AUC-ROC por patógeno (LOPO)'); plt.xlim(0,1)
plt.legend(); plt.tight_layout(); savefig('fig3_auc_por_patogeno'); plt.show()

# Fig 4 - Per-pathogen diagnostic, homology-aware vs LOPO
d = cmp.dropna(subset=['auc_lopo','auc_hom']).sort_values('auc_lopo')
yy = np.arange(len(d)); h = 0.4
plt.figure(figsize=(8, max(3, 0.55*len(d))))
plt.barh(yy+h/2, d['auc_hom'],  height=h, label='Homology-aware (patógeno visto)', color='#4C9F70')
plt.barh(yy-h/2, d['auc_lopo'], height=h, label='LOPO (patógeno fuera)',           color='#1f77b4')
plt.yticks(yy, d['group']); plt.axvline(0.5, color='gray', ls='--'); plt.xlim(0,1)
plt.xlabel('AUC-ROC'); plt.title('AUC por patógeno: homology-aware vs LOPO')
plt.legend(loc='lower right'); plt.tight_layout(); savefig('fig4_diagnostico_hom_vs_lopo'); plt.show()

# Fig 5 - Metric bars (nested vs LOPO)
ax = summary.T.plot(kind='bar', figsize=(9,4.5), rot=0, width=.8)
ax.set_ylim(min(0.0, float(summary.values.min()))-0.05, 1.05); ax.set_ylabel('Valor')
ax.set_title('Métricas OOF: nested homology-aware vs LOPO'); ax.legend(title=None, loc='lower right')
for c in ax.containers: ax.bar_label(c, fmt='%.2f', fontsize=8, padding=2)
plt.tight_layout(); savefig('fig5_metricas'); plt.show()

print('Figuras guardadas en', FIGDIR)

## Notas

- **Modelo final:** sus hiperparámetros se eligen sobre TODO el dataset (celda 7). No se reporta una métrica "de test" suya porque no se reserva test: su rendimiento esperado son los números del **nested CV** (rendimiento del método) y del **LOPO** (generalización a patógeno nuevo).
- **Umbral 0.5:** precision/recall/f1/mcc usan 0.5. AUC-ROC y AUC-PR no dependen del umbral. Si quieres, ajusta el umbral maximizando MCC/F1 en la OOF antes de reportar precision/recall.
- **Ratio de clases:** el pipeline es agnóstico al balance (usa `class_weight='balanced'`). Si cambias `embeddings.npz` por una versión 1:N, funciona igual.
- **LOPO totalmente anidado (opción rigurosa):** hoy el LOPO usa `final_cfg` (fija). Para re-buscar hiperparámetros dentro de cada patógeno, sustituye en `lopo_fixed` la línea del `cfg` por `cfg,_ = inner_search(X[tr], y[tr], folds[tr], N_TRIALS_NESTED)` (≈16 búsquedas más).
- **CSVs:** `oof_predictions.csv` (una fila por muestra: scheme, id, y_true, y_score, pathogen, homology_fold) es el que te deja rehacer cualquier figura conjunta. Los demás son métricas agregadas.
